In [2]:
import pandas as pd
from pathlib import Path

fichier = Path(r"C:\Users\OYONO\Documents\COURS DE FORMATION\0000 PROJET FIN DE FORMATION DATA ANALYST IA\PROJET_FINAL_BTP_NANTES\dpe_nantes.csv")

df = pd.read_csv(fichier)

print("Nombre total de diagnostics :", len(df))
print("Surfaces manquantes :", df["surface_habitable_logement"].isna().sum())

Nombre total de diagnostics : 133234
Surfaces manquantes : 3026


In [3]:
print("Surfaces égales à zéro :", (df["surface_habitable_logement"] == 0).sum())

print("Surfaces négatives :", (df["surface_habitable_logement"] < 0).sum())

Surfaces égales à zéro : 0
Surfaces négatives : 0


In [4]:
print(df["surface_habitable_logement"].describe())

count    130208.000000
mean         58.411144
std          56.548983
min           6.700000
25%          36.000000
50%          55.100000
75%          71.400000
max        7759.600000
Name: surface_habitable_logement, dtype: float64


In [5]:
colonnes = [
    "numero_dpe",
    "type_batiment",
    "surface_habitable_logement"
]

print(df.nlargest(5, "surface_habitable_logement")[colonnes])


          numero_dpe type_batiment  surface_habitable_logement
12892  2244E0051260H      immeuble                      7759.6
11689  2244E0053908D      immeuble                      6441.2
5632   2144E0695380C      immeuble                      6249.0
7160   2144E0715454E      immeuble                      6249.0
4759   2144E0594693N      immeuble                      5912.0


In [6]:
print(df["type_batiment"].value_counts(dropna=False))


type_batiment
appartement    118888
maison          11323
immeuble         3023
Name: count, dtype: int64


In [7]:
print(
    df["etiquette_dpe"]
    .value_counts(dropna=False)
    .sort_index()
)

etiquette_dpe
A      769
B     7811
C    62331
D    39790
E    15272
F     5020
G     2241
Name: count, dtype: int64


In [8]:

tableau = pd.crosstab(
    df["type_batiment"],
    df["etiquette_dpe"]
)

display(tableau)

etiquette_dpe,A,B,C,D,E,F,G
type_batiment,,,,,,,
appartement,616,7448,57844,34486,12696,3962,1836
immeuble,17,135,1194,969,475,178,55
maison,136,228,3293,4335,2101,880,350


In [9]:
tableau_pct = pd.crosstab(
    df["type_batiment"],
    df["etiquette_dpe"],
    normalize="index"
) * 100

display(tableau_pct.round(2))

etiquette_dpe,A,B,C,D,E,F,G
type_batiment,,,,,,,
appartement,0.52,6.26,48.65,29.01,10.68,3.33,1.54
immeuble,0.56,4.47,39.50,32.05,15.71,5.89,1.82
maison,1.20,2.01,29.08,38.28,18.56,7.77,3.09


In [10]:
print("Diagnostics :", len(df))

print("Numéros DPE distincts :", df["numero_dpe"].nunique())

print("Numéros DPE manquants :", df["numero_dpe"].isna().sum())

Diagnostics : 133234
Numéros DPE distincts : 133234
Numéros DPE manquants : 0


In [11]:
print("Identifiants BAN manquants :",
      df["identifiant_ban"].isna().sum())

print("Identifiants BAN distincts :",
      df["identifiant_ban"].nunique())


Identifiants BAN manquants : 13
Identifiants BAN distincts : 20739


In [12]:
diagnostics_par_adresse = (
    df["identifiant_ban"]
    .value_counts()
    .head(10)
)

display(diagnostics_par_adresse)

identifiant_ban
44109_4552_00032    472
44109_7218          359
44109_5970_00011    304
44109_3377_00021    303
44109_5636_00075    295
44109_3797_00002    286
44109_3680_00039    280
44109_0852_00024    273
44109_4152_00009    266
44109_7809_00006    264
Name: count, dtype: int64

In [13]:
display(
    df.loc[
        df["type_batiment"] == "immeuble",
        ["numero_dpe", "type_batiment",
         "surface_habitable_logement"]
    ].head(5)
)

,numero_dpe,type_batiment,surface_habitable_logement
16,2644E0045752P,immeuble,NaN
19,2644E0008167A,immeuble,NaN
23,2644E0074297M,immeuble,NaN
69,2644E0075755O,immeuble,NaN
101,2644E0011880V,immeuble,NaN


In [14]:
print(
    df.groupby("type_batiment")["surface_habitable_logement"]
    .agg(
        diagnostics="size",
        surfaces_manquantes=lambda x: x.isna().sum()
    )
)

               diagnostics  surfaces_manquantes
type_batiment                                  
appartement         118888                   21
immeuble              3023                 3005
maison               11323                    0


In [15]:
immeubles_surface = df.loc[
    (df["type_batiment"] == "immeuble")
    & (df["surface_habitable_logement"].notna()),
    ["numero_dpe", "surface_habitable_logement"]
]

display(immeubles_surface)

,numero_dpe,surface_habitable_logement
1852,2144E0232602Y,4993.0
3773,2144E0821665F,1879.0
4150,2144E0715125N,956.0
4449,2144E0829118W,2151.4
4734,2144E0929811R,1770.9
4759,2144E0594693N,5912.0
5467,2144E0739018M,1857.0
5568,2144E0828979N,2458.5
5632,2144E0695380C,6249.0
5689,2144E0827029N,1179.7


In [16]:

print(
    df.groupby("type_batiment")["nombre_niveau_logement"]
    .agg(
        diagnostics="size",
        valeurs_renseignees="count"
    )
)

               diagnostics  valeurs_renseignees
type_batiment                                  
appartement         118888               118686
immeuble              3023                   16
maison               11323                11323


In [41]:
resultat = df.loc[
    (df["type_batiment"] == "immeuble")
    & (df["nombre_niveau_logement"].notna()),
    [
        "numero_dpe",
        "surface_habitable_logement",
        "nombre_niveau_logement"
    ]
]

display(resultat)

,numero_dpe,surface_habitable_logement,nombre_niveau_logement
3773,2144E0821665F,1879.0,4.0
4150,2144E0715125N,956.0,5.0
4449,2144E0829118W,2151.4,5.0
4734,2144E0929811R,1770.9,8.0
5467,2144E0739018M,1857.0,5.0
5568,2144E0828979N,2458.5,5.0
5632,2144E0695380C,6249.0,7.0
5689,2144E0827029N,1179.7,5.0
6505,2144E0827191T,1029.1,5.0
6666,2144E0886442Q,2978.3,7.0


In [18]:
adresses_immeubles = set(
    df.loc[
        df["type_batiment"] == "immeuble",
        "identifiant_ban"
    ].dropna()
)

adresses_appartements = set(
    df.loc[
        df["type_batiment"] == "appartement",
        "identifiant_ban"
    ].dropna()
)

adresses_communes = adresses_immeubles & adresses_appartements

print("Adresses communes :", len(adresses_communes))

Adresses communes : 2220


In [19]:
immeubles_associes = df.loc[
    (df["type_batiment"] == "immeuble")
    & (df["identifiant_ban"].isin(adresses_communes))
]

print("Diagnostics immeubles associés :", len(immeubles_associes))


Diagnostics immeubles associés : 2886


In [20]:
immeubles_restants = df.loc[
    (df["type_batiment"] == "immeuble")
    & (~df["identifiant_ban"].isin(adresses_communes))
]

print("Diagnostics restants :", len(immeubles_restants))
print("Identifiants BAN manquants :",
      immeubles_restants["identifiant_ban"].isna().sum())

Diagnostics restants : 137
Identifiants BAN manquants : 0


In [21]:
print(
    df["periode_construction"]
    .value_counts(dropna=False)
    .sort_index()
)

periode_construction
1948-1974     44968
1975-1977      3968
1978-1982      4767
1983-1988      5276
1989-2000     15911
2001-2005      3908
2006-2012     13450
2013-2021     12432
après 2021     2292
avant 1948    26262
Name: count, dtype: int64


In [22]:
tableau_periode = pd.crosstab(
    df["periode_construction"],
    df["etiquette_dpe"],
    normalize="index"
) * 100

display(tableau_periode.round(2))

etiquette_dpe,A,B,C,D,E,F,G
periode_construction,,,,,,,
1948-1974,0.16,5.17,32.12,40.37,15.91,4.90,1.36
1975-1977,0.00,4.89,59.10,26.01,9.25,0.68,0.08
1978-1982,0.02,1.26,53.12,38.10,6.50,0.59,0.42
1983-1988,0.00,1.02,60.39,34.33,3.73,0.36,0.17
1989-2000,0.03,1.46,69.20,26.74,2.34,0.19,0.04
2001-2005,0.08,5.48,79.40,14.05,0.90,0.05,0.05
2006-2012,0.25,9.23,75.54,14.16,0.77,0.03,0.02
2013-2021,4.29,22.39,71.47,1.77,0.06,0.03,0.00
après 2021,3.71,22.69,71.95,1.61,0.04,0.00,0.00


In [23]:
diagnostics_efg = df[
    df["etiquette_dpe"].isin(["E", "F", "G"])
]

resultat_efg = (
    diagnostics_efg["periode_construction"]
    .value_counts()
    .sort_index()
)

display(resultat_efg)

periode_construction
1948-1974      9970
1975-1977       397
1978-1982       358
1983-1988       225
1989-2000       408
2001-2005        39
2006-2012       110
2013-2021        11
après 2021        1
avant 1948    11014
Name: count, dtype: int64

In [24]:

tableau_cible = pd.crosstab(
    diagnostics_efg["periode_construction"],
    diagnostics_efg["type_batiment"]
)

display(tableau_cible)


type_batiment,appartement,immeuble,maison
periode_construction,,,
1948-1974,8343,297,1330
1975-1977,354,9,34
1978-1982,295,11,52
1983-1988,166,3,56
1989-2000,351,12,45
2001-2005,31,1,7
2006-2012,90,7,13
2013-2021,9,0,2
après 2021,1,0,0


In [25]:
surfaces_efg = diagnostics_efg[
    diagnostics_efg["type_batiment"].isin(["maison", "appartement"])
]

display(
    surfaces_efg.groupby("type_batiment")[
        "surface_habitable_logement"
    ].agg(["count", "mean", "median", "min", "max"]).round(2)
)


,count,mean,median,min,max
type_batiment,,,,,
appartement,18484,48.58,44.5,6.7,462.0
maison,3331,93.21,83.9,12.0,638.6


In [26]:
print(
    surfaces_efg.groupby("type_batiment")[
        "surface_habitable_logement"
    ].apply(lambda x: (x > 200).sum())
)

type_batiment
appartement     42
maison         107
Name: surface_habitable_logement, dtype: int64


In [27]:
surfaces_efg = surfaces_efg.copy()

surfaces_efg["classe_surface"] = pd.cut(
    surfaces_efg["surface_habitable_logement"],
    bins=[0, 50, 100, 150, 200, float("inf")],
    labels=["S1", "S2", "S3", "S4", "S5"],
    right=False
)

display(
    pd.crosstab(
        surfaces_efg["classe_surface"],
        surfaces_efg["type_batiment"]
    )
)

type_batiment,appartement,maison
classe_surface,,
S1,10762,504
S2,7121,1609
S3,474,880
S4,85,230
S5,42,108


In [28]:
print(
    surfaces_efg.loc[
        surfaces_efg["surface_habitable_logement"] == 200,
        "type_batiment"
    ].value_counts()
)

type_batiment
maison    1
Name: count, dtype: int64


In [29]:
print(
    df["type_energie_principale_chauffage"]
    .value_counts(dropna=False)
)


type_energie_principale_chauffage
Électricité                                                     50658
Gaz naturel                                                     49945
Réseau de Chauffage urbain                                      31847
Fioul domestique                                                  344
GPL                                                               188
Bois – Bûches                                                     149
Bois – Granulés (pellets) ou briquettes                            96
Bois – Plaquettes d’industrie                                       2
Butane                                                              2
Électricité d'origine renouvelable utilisée dans le bâtiment        1
Charbon                                                             1
Non affecté                                                         1
Name: count, dtype: int64


In [30]:
tableau_chauffage = pd.crosstab(
    df["type_energie_principale_chauffage"],
    df["etiquette_dpe"]
)

display(tableau_chauffage)

etiquette_dpe,A,B,C,D,E,F,G
type_energie_principale_chauffage,,,,,,,
Bois – Bûches,11,24,51,30,21,1,11
Bois – Granulés (pellets) ou briquettes,9,9,48,20,8,2,0
Bois – Plaquettes d’industrie,0,0,1,0,0,0,1
Butane,0,0,0,1,0,0,1
Charbon,0,0,0,0,0,0,1
Fioul domestique,0,0,7,36,96,104,101
GPL,0,0,67,73,29,17,2
Gaz naturel,6,618,21904,16980,7653,2247,537
Non affecté,1,0,0,0,0,0,0


In [31]:
taux_efg = (
    tableau_chauffage[["E", "F", "G"]].sum(axis=1)
    / tableau_chauffage.sum(axis=1)
    * 100
)

display(taux_efg.round(2).sort_values(ascending=False))


type_energie_principale_chauffage
Charbon                                                         100.00
Fioul domestique                                                 87.50
Butane                                                           50.00
Bois – Plaquettes d’industrie                                    50.00
GPL                                                              25.53
Bois – Bûches                                                    22.15
Gaz naturel                                                      20.90
Électricité                                                      19.20
Bois – Granulés (pellets) ou briquettes                          10.42
Réseau de Chauffage urbain                                        6.20
Non affecté                                                       0.00
Électricité d'origine renouvelable utilisée dans le bâtiment      0.00
dtype: float64

In [32]:
synthese_chauffage = pd.DataFrame({
    "total_diagnostics": tableau_chauffage.sum(axis=1),
    "diagnostics_efg": tableau_chauffage[["E", "F", "G"]].sum(axis=1),
    "taux_efg": taux_efg.round(2)
})

display(
    synthese_chauffage.sort_values(
        "diagnostics_efg", ascending=False
    )
)

,total_diagnostics,diagnostics_efg,taux_efg
type_energie_principale_chauffage,,,
Gaz naturel,49945,10437,20.90
Électricité,50658,9726,19.20
Réseau de Chauffage urbain,31847,1975,6.20
Fioul domestique,344,301,87.50
GPL,188,48,25.53
Bois – Bûches,149,33,22.15
Bois – Granulés (pellets) ou briquettes,96,10,10.42
Bois – Plaquettes d’industrie,2,1,50.00
Butane,2,1,50.00


In [33]:
colonnes_sql = [
    "numero_dpe",
    "identifiant_ban",
    "type_batiment",
    "etiquette_dpe",
    "surface_habitable_logement",
    "periode_construction",
    "type_energie_principale_chauffage"
]

df_sql = df[colonnes_sql].copy()

df_sql.to_csv(
    "FactDPE_import.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Fichier préparé :", len(df_sql), "diagnostics")

Fichier préparé : 133234 diagnostics


In [34]:
df_sql.to_csv(
    "FactDPE_import.csv",
    index=False,
    encoding="utf-8"
)

In [35]:
[col for col in df.columns if "date" in col.lower()]

['date_etablissement_dpe', 'date_reception_dpe']

In [36]:
df["date_etablissement_dpe"].head(5)

0    2026-01-07
1    2026-01-07
2    2026-01-08
3    2025-07-30
4    2026-01-05
Name: date_etablissement_dpe, dtype: str

In [37]:

df["date_etablissement_dpe"] = pd.to_datetime(
    df["date_etablissement_dpe"],
    errors="coerce"
)


In [38]:
dim_date = pd.DataFrame({
    "date_dpe": df["date_etablissement_dpe"].dropna().unique()
})

dim_date["date_dpe"] = pd.to_datetime(dim_date["date_dpe"])
dim_date["annee"] = dim_date["date_dpe"].dt.year
dim_date["trimestre"] = dim_date["date_dpe"].dt.quarter
dim_date["mois"] = dim_date["date_dpe"].dt.month

dim_date = dim_date.sort_values("date_dpe").reset_index(drop=True)

In [39]:
dim_date.to_csv(
    "DimDate.csv",
    index=False,
    encoding="utf-8",
    date_format="%Y-%m-%d"
)

In [40]:
df[["numero_dpe", "date_etablissement_dpe"]].to_csv(
    "FactDPE_Dates.csv",
    index=False,
    encoding="utf-8",
    date_format="%Y-%m-%d"
)


## Mission 3 — Journal de qualité des données DPE

| Contrôle | Constat | Décision | Lignes modifiées |
|---|---|---|---:|
| Numéros DPE | 133 234 uniques, 0 manquant | Conserver | 0 |
| Identifiants BAN | 13 manquants | Conserver les valeurs manquantes | 0 |
| Surfaces habitables | 3 026 manquantes | Conserver les valeurs NULL | 0 |
| Surfaces nulles ou négatives | Aucune | Aucune correction | 0 |
| Grandes surfaces | Principalement des immeubles | Conserver et analyser séparément | 0 |
| Niveaux des immeubles | 16 renseignés sur 3 023 | Signaler l'incomplétude | 0 |

| Types de données | Types des principales colonnes cohérents | Conserver les types actuels | 0 |

| Normalisation du type de bâtiment | 3 catégories cohérentes : appartement, immeuble, maison | Conserver les valeurs | 0 |

| Classes énergétiques DPE | 7 classes A à G, aucune valeur manquante | Conserver les valeurs | 0 |

| Énergie principale de chauffage | 13 catégories, aucune valeur manquante | Conserver les catégories d'origine | 0 |

| Périodes de construction | 11 catégories, aucune valeur manquante | Conserver les périodes d'origine | 0 |

| Coordonnées géographiques X/Y | 0 valeur manquante sur les deux colonnes | Conserver ; vérifier le système de coordonnées avant la jointure spatiale | 0 |



In [42]:
print(df.dtypes)

numero_dpe                                      str
date_etablissement_dpe               datetime64[us]
date_reception_dpe                              str
modele_dpe                                      str
numero_dpe_remplace                             str
etiquette_dpe                                   str
etiquette_ges                                   str
conso_5_usages_par_m2_ep                    float64
conso_5_usages_par_m2_ef                    float64
emission_ges_5_usages_par_m2                float64
cout_total_5_usages                         float64
type_batiment                                   str
annee_construction                          float64
periode_construction                            str
surface_habitable_logement                  float64
numero_etage_appartement                    float64
nombre_niveau_logement                      float64
type_energie_principale_chauffage               str
type_installation_chauffage                     str
adresse_ban 

In [43]:
print(df["type_batiment"].unique())

<StringArray>
['appartement', 'immeuble', 'maison']
Length: 3, dtype: str


In [44]:
print(df["etiquette_dpe"].value_counts(dropna=False))

etiquette_dpe
C    62331
D    39790
E    15272
B     7811
F     5020
G     2241
A      769
Name: count, dtype: int64


In [45]:
print(df["type_energie_principale_chauffage"].value_counts(dropna=False))

type_energie_principale_chauffage
Électricité                                                     50658
Gaz naturel                                                     49945
Réseau de Chauffage urbain                                      31847
Fioul domestique                                                  344
GPL                                                               188
Bois – Bûches                                                     149
Bois – Granulés (pellets) ou briquettes                            96
Bois – Plaquettes d’industrie                                       2
Butane                                                              2
Électricité d'origine renouvelable utilisée dans le bâtiment        1
Charbon                                                             1
Non affecté                                                         1
Name: count, dtype: int64


In [46]:
print(df["periode_construction"].value_counts(dropna=False))

periode_construction
1948-1974     44968
avant 1948    26262
1989-2000     15911
2006-2012     13450
2013-2021     12432
1983-1988      5276
1978-1982      4767
1975-1977      3968
2001-2005      3908
après 2021     2292
Name: count, dtype: int64


In [49]:
print(
    df[
        [
            "coordonnee_cartographique_x_ban",
            "coordonnee_cartographique_y_ban"
        ]
    ].isna().sum()
)

coordonnee_cartographique_x_ban    0
coordonnee_cartographique_y_ban    0
dtype: int64


In [50]:
print(
    df[
        [
            "coordonnee_cartographique_x_ban",
            "coordonnee_cartographique_y_ban"
        ]
    ].head(5)
)

   coordonnee_cartographique_x_ban  coordonnee_cartographique_y_ban
0                        358620.37                       6694345.12
1                        357210.47                       6690360.17
2                        358174.01                       6691584.26
3                        359110.42                       6691873.68
4                        354557.61                       6688183.23


In [51]:
import geopandas as gpd

quartiers = gpd.read_file(
    "244400404_quartiers-de-la-ville-de-nantes.geojson"
)

print("Système de coordonnées :", quartiers.crs)
print("Nombre de quartiers :", len(quartiers))

ModuleNotFoundError: No module named 'geopandas'

In [53]:
import geopandas as gpd

quartiers = gpd.read_file(
    "244400404_quartiers-de-la-ville-de-nantes.geojson"
)

print("Système de coordonnées :", quartiers.crs)
print("Nombre de quartiers :", len(quartiers))

Système de coordonnées : EPSG:4326
Nombre de quartiers : 11


In [54]:
quartiers_2154 = quartiers.to_crs(epsg=2154)

print("Système après conversion :", quartiers_2154.crs)
print("Nombre de quartiers :", len(quartiers_2154))

Système après conversion : EPSG:2154
Nombre de quartiers : 11


In [55]:
dpe_geo = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(
        df["coordonnee_cartographique_x_ban"],
        df["coordonnee_cartographique_y_ban"]
    ),
    crs="EPSG:2154"
)

print("Nombre de points DPE :", len(dpe_geo))
print("Système de coordonnées :", dpe_geo.crs)

Nombre de points DPE : 133234
Système de coordonnées : EPSG:2154


In [56]:
dpe_quartiers = gpd.sjoin(
    dpe_geo,
    quartiers_2154,
    how="left",
    predicate="within"
)

print("Diagnostics initiaux :", len(dpe_geo))
print("Lignes après jointure :", len(dpe_quartiers))
print("Diagnostics sans quartier :", dpe_quartiers["index_right"].isna().sum())


Diagnostics initiaux : 133234
Lignes après jointure : 133234
Diagnostics sans quartier : 50


In [52]:
%pip install geopandas

   ---------------------------------------- 0.0/23.8 MB ? eta -:--:--
   ------------ --------------------------- 7.6/23.8 MB 37.4 MB/s eta 0:00:01
   --------------------------- ------------ 16.3/23.8 MB 39.1 MB/s eta 0:00:01
   ---------------------------------------- 23.8/23.8 MB 37.2 MB/s  0:00:00
   ---------------------------------------- 0.0/6.9 MB ? eta -:--:--
   ---------------------------------------- 6.9/6.9 MB 42.0 MB/s  0:00:00
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 41.4 MB/s  0:00:00

   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   -


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [57]:
print(quartiers_2154.columns.tolist())

['nom', 'libcom', 'codcom', 'geo_point_2d', 'geometry']


In [58]:
print(quartiers_2154["nom"].tolist())

['Île de Nantes', 'Nantes Sud', 'Doulon - Bottière', 'Nantes Nord', 'Breil - Barberie', 'Centre-ville', 'Hauts-Pavés - Saint-Félix', 'Dervallières - Zola', 'Bellevue - Chantenay - Sainte-Anne', 'Malakoff - Saint-Donatien', 'Nantes Erdre']


In [59]:
correspondance = dict(
    zip(
        range(1, 12),
        quartiers_2154["nom"]
    )
)

print(correspondance)

{1: 'Île de Nantes', 2: 'Nantes Sud', 3: 'Doulon - Bottière', 4: 'Nantes Nord', 5: 'Breil - Barberie', 6: 'Centre-ville', 7: 'Hauts-Pavés - Saint-Félix', 8: 'Dervallières - Zola', 9: 'Bellevue - Chantenay - Sainte-Anne', 10: 'Malakoff - Saint-Donatien', 11: 'Nantes Erdre'}


In [60]:
nom_vers_id = {
    nom: identifiant
    for identifiant, nom in correspondance.items()
}

dpe_quartiers["id_quartier"] = (
    dpe_quartiers["nom"].map(nom_vers_id)
)

print("Diagnostics avec quartier :",
      dpe_quartiers["id_quartier"].notna().sum())

print("Diagnostics sans quartier :",
      dpe_quartiers["id_quartier"].isna().sum())

Diagnostics avec quartier : 133184
Diagnostics sans quartier : 50


In [61]:
dpe_quartiers[
    ["numero_dpe", "id_quartier"]
].to_csv(
    "DPE_Quartiers_import.csv",
    index=False,
    encoding="utf-8"
)

print("Fichier créé : DPE_Quartiers_import.csv")

Fichier créé : DPE_Quartiers_import.csv
